In [1]:
#Array packages
import pandas as pd
import numpy as np
import xarray as xr
import netCDF4 as nc4

from scipy.stats import kendalltau
import pymannkendall as mk

#plots
import matplotlib.pyplot as plt
import rioxarray as rio
import geopandas as gpd
from shapely.geometry import mapping
import cartopy.crs as ccrs

#Progress meter
from dask.diagnostics import ProgressBar
from tqdm import tqdm

# Directories
import os
import glob
import dask
#import h5netcdf
import scipy

import os
#os.chdir(r"E:\academy\OneDrive - IIT Delhi\3. IIT DELHI\2. Research\2_Papers\1_Clustering connectivity")
os.chdir(r"G:\OneDrive - IIT Delhi\3. IIT DELHI\2. Research\2_Papers\1_Clustering connectivity")
print(os.getcwd())

G:\OneDrive - IIT Delhi\3. IIT DELHI\2. Research\2_Papers\1_Clustering connectivity


### 1. DATA preprocessing

### 1.1 Change the start data and end data column of metadata file based on streamflow timeseries (initial it was based on stream level)   

In [ ]:
from shapely.geometry import Point

A=pd.read_csv(f'3_Data/Data_p/2_Station/1_Streamflow_data/gauge_info_p.csv')  
south_asia=gpd.read_file(r'3_Data\Data_p\3_Shapefiles\south_asia_p.shp')
south_asia.set_index('Basin',inplace=True)

south_asia


geometry = [Point(lon, lat) for lon, lat in zip(A['Longitude'], A['Latitude'])]
stations_gdf = gpd.GeoDataFrame(A, geometry=geometry, crs=south_asia.crs)
stations_gdf 


# Perform spatial join to classify stations into watersheds
stations_classified = gpd.sjoin(stations_gdf, south_asia, how='left', predicate='within')

# Rename the column containing watershed names for clarity
stations_classified = stations_classified.rename(columns={'Watershed_Name': 'Assigned_Watershed'})


stations_classified

,Unnamed: 0,GaugeID,Station,Latitude,Longitude,River Name/ Tributory/ SubTributory,Basin,State,Start_date,End_date,Streamflow_Entries,Privacy,Expected_entries,missing_percent,geometry,index_right,SUB_AREA,UP_AREA
0,0,IWM-gauge-1,Thotapalli,18.7800,83.5000,NaN,East flowing rivers between mahanadi and pennar,Andhra Pradesh,1999-08-01,2015-07-07,5727,Open,5820.0,1.597938,POINT (83.50000 18.78000),EFMP,NaN,NaN
1,2,IWM-gauge-3,Kattaleru @ tiruvuru,17.0900,80.6200,NaN,Krishna,Andhra Pradesh,1999-07-28,2014-11-06,1216,Open,5581.0,78.211790,POINT (80.62000 17.09000),Krishna,9181.9,135485.3
2,3,IWM-gauge-4,Nandipalli,14.7211,79.0189,Pennar/Sagaileru,Pennar,Andhra Pradesh,1990-06-01,2018-11-12,9712,Open,10392.0,6.543495,POINT (79.01890 14.72110),Cauvery,54905.1,54905.1
3,4,IWM-gauge-5,Paradespalem,17.8500,83.3600,NaN,East flowing rivers between mahanadi and pennar,Andhra Pradesh,1999-10-01,2005-12-03,404,Open,2256.0,82.092199,POINT (83.36000 17.85000),EFMP,NaN,NaN
4,6,IWM-gauge-7,Pillaperu riverr @ narravada.,14.9300,79.4200,NaN,East flowing rivers between mahanadi and pennar,Andhra Pradesh,2000-08-31,2018-07-08,877,Open,6521.0,86.551142,POINT (79.42000 14.93000),EFMP,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
630,1080,IWM-gauge-1081,Fekoghat,22.3086,86.9164,Subarnarekha/Dul ung,Subarnarekha,West Bengal,1988-06-18,2018-10-31,4000,Open,11093.0,63.941224,POINT (86.91640 22.30860),Subarnarekha,NaN,NaN
631,1081,IWM-gauge-1082,Gopiballavpur,22.2200,86.9000,Subarnarekha,Subernarekha,West Bengal,2015-06-04,2019-03-06,1372,Open,1372.0,0.000000,POINT (86.90000 22.22000),Subarnarekha,NaN,NaN
632,1101,IWM-gauge-1102,Domohani,26.5628,88.7583,Brahmaputra/ Teesta,Ganga - Brahmaputra -Meghna/Barak,West Bengal,2000-01-06,2021-05-31,7628,Restricted,7817.0,2.417807,POINT (88.75830 26.56280),Brahmaputra,NaN,NaN
633,1102,IWM-gauge-1103,Hasimara,26.7292,89.3236,Brahmaputra/ Torsa,Ganga - Brahmaputra -Meghna/Barak,West Bengal,1990-01-06,2021-05-31,11296,Restricted,11469.0,1.508414,POINT (89.32360 26.72920),Brahmaputra,NaN,NaN


## 2. station subsetting
 1. Identify the stations with minimum 30 years of data
 2. Subset the station with minimum 30 years of data and with period upto 2010
 3. Percentage of missing data

### 2.1 Full stations

# FUNCTIONS

### 1. Station visualization

In [6]:

def Station_Map(data,condition):

    #data=pd.read_csv(r"3_Data\Data_p\2_Station\1_Streamflow_data\data.csv")
    south_asia=gpd.read_file(r'3_Data\Data_p\3_Shapefiles\south_asia_p.shp')
 
    fig, ax1 = plt.subplots(1, 1, figsize=(6,6), subplot_kw={"projection": ccrs.PlateCarree()})
    south_asia.plot(ax=ax1,color = 'none',edgecolor = 'black',linewidth=0.55,alpha=0.9)

    #Axis setting
    [x.set_visible(False) for x in ax1.spines.values()]
    [x.set_linewidth(0.2) for x in ax1.spines.values()]

    c1=np.repeat(['Blue'],len(data['Latitude']))
    sc=ax1.scatter(data['Longitude'],data['Latitude'],c='firebrick',s=13,edgecolor='black')

    ax1.text(0.50, 0.95, f'Stations : {condition}', fontsize=12, color='black',fontname='Times New Roman',weight='bold',
    transform=ax1.transAxes, ha='left', va='center')

    ax1.text(0.5, 0.9, f'No of Station : {len(data)}' , fontsize=12, color='black',fontname='Times New Roman',weight='bold',
    transform=ax1.transAxes, ha='left', va='center')

    plt.savefig(fr'2_Analysis\0_Datapreprocessing\Outputs\{condition}', bbox_inches='tight',dpi=1000)
